In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [2]:
# Select the t5 small model
tokenizer = T5Tokenizer.from_pretrained('t5-small') # Tokenizer: text -> tokens
model = T5ForConditionalGeneration.from_pretrained('t5-small') # model that generates summaries

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
from datasets import load_dataset
train_ds = load_dataset('xsum', split='train[:1000]') # import the first 1k training set items
val_ds = load_dataset('xsum', split='validation[:200]') # import the first 200 validation set items

In [4]:
train_ds

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 1000
})

In [5]:
val_ds

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 200
})

In [6]:
train_ds[0]

{'document': 'The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally apprecia

In [7]:
def preprocess(example): # Example is a dict containing the document, its summary label and an id
  text = 'summarize: ' + example['document'] # Add the summarize command
  inputs_document = tokenizer(text, max_length=512, truncation=True) # Tokenize the text
  inputs_label = tokenizer(example['summary'], max_length=512, truncation=True) # Tokenize the target

  preprocessed = {
      'input_ids' : inputs_document['input_ids'],
      'attention_mask' : inputs_document['attention_mask'],
      'labels' : inputs_label['input_ids']
  }

  return preprocessed

In [24]:
# Preprocess the datasets (add 3 extra features in the dicts input_ids, attention_mask, labels)
train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [38]:
train_ds[0].keys()

dict_keys(['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'])

In [25]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model) # Data Collator is used to add padding to the batch, to match the longest item

In [26]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [27]:
training_args = Seq2SeqTrainingArguments( # Setup the hyperparams of the model
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    eval_strategy='epoch',
    predict_with_generate=True,
)

In [28]:
trainer = Seq2SeqTrainer( # Set up the model trainer
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.789510
2,No log,2.750127
3,No log,2.742230


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=2.9956227213541666, metrics={'train_runtime': 128.7551, 'train_samples_per_second': 23.3, 'train_steps_per_second': 2.913, 'total_flos': 405596117139456.0, 'train_loss': 2.9956227213541666, 'epoch': 3.0})

In [31]:
import torch

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
# Check out an example
example = val_ds[0]
inputs = tokenizer.decode(example['input_ids'], skip_special_tokens=True)
print("Input:", inputs[:200])
print("---")
outputs = model.generate(torch.tensor([example['input_ids']]).to(model.device), max_length=60, num_beams=4)
print("Fine-tuned:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Input: summarize: The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport. Mr Sodje, 37, is jointly charged with elder 
---
Fine-tuned: Sam Sodje, 37, and Stephen Sodje, 37, are accused of fraudulent trading charges relating to the Sodje Sports Foundation.


In [35]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=9519832ab53287a16eae6639f02d8db82be69009b8c371b897128c63ad2186c7
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [36]:
from rouge_score import rouge_scorer

In [37]:
scorer = rouge_scorer.RougeScorer(rouge_types=['rouge1', 'rouge2', 'rougeL'])

# Function to compute the 3 different rouge scores
def compute_rouge(prediction, reference):
  scores = scorer.score(prediction, reference)
  return scores

In [45]:
# Test out on the first 5 examples from the val_set and compute the rouge scores

avg_f1_score_rouge1 = 0
avg_f1_score_rouge2 = 0
avg_f1_score_rougeL = 0

for i in range(len(val_ds)):
  val_example = val_ds[i]['document']
  val_reference = val_ds[i]['summary']
  val_inputs = tokenizer('summarize: ' + val_example, return_tensors='pt', max_length=512, truncation=True).to(model.device)
  val_output = model.generate(val_inputs['input_ids'], max_length=50)
  val_summary = tokenizer.decode(val_output[0], skip_special_tokens=True)
  results = compute_rouge(val_summary, val_reference)
  avg_f1_score_rouge1 += results['rouge1'].fmeasure
  avg_f1_score_rouge2 += results['rouge2'].fmeasure
  avg_f1_score_rougeL += results['rougeL'].fmeasure

print(f'Average Rouge1 F1: {avg_f1_score_rouge1 / len(val_ds)}')
print(f'Average Rouge2 F1: {avg_f1_score_rouge2 / len(val_ds)}')
print(f'Average RougeL F1: {avg_f1_score_rougeL / len(val_ds)}')

Average Rouge1 F1: 0.22036655350217582
Average Rouge2 F1: 0.05055352494548738
Average RougeL F1: 0.17317310252313167
